# Milestone 6 Lab, notebook 1 of 8: Late and corrected files, by hand

## Where we are

| Notebook | What you do |
|---|---|
| **1. Late and corrected files, by hand** (you are here) | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **0**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

Until now every file arrived once, complete, and on time. This is the last assumption left to remove.

In this notebook three files arrive **after the month was closed and reported**. You work out by hand, with no AI, what each one should do to the figures. That gives you an answer key for the rest of the milestone.

> If you have not done milestones 1 to 5, do them first.

In [ ]:
import pandas as pd
from lab_helpers import LAB_E2E, profit_of, save_result, show_doc

## Step 1: What the business is asking for now

In [ ]:
show_doc("BRD.md", "1. Business problem")

In [ ]:
show_doc("BRD.md", "5. Words used in this milestone")

## Step 2: Build the system up to month-end close

`build_close` does all of milestone 5 in one call, from nothing: it writes the fifty files, loads them, and makes `RUN-0001` and `RUN-0002`. It uses a folder of the lab's own, and takes about a second. No model is called.

In [ ]:
from retail_profit_stack.e2e import build_close

workspace = build_close(LAB_E2E)

print("Files loaded:", len(list(workspace.places.landing.glob("*.csv"))))
print("Runs:", sorted(folder.name for folder in (workspace.warehouse / "runs").iterdir()))

This state is called **month-end close**. September and October have been calculated and reported. Here are three figures as they stand at close. Remember them.

In [ ]:
at_close = {
    "Fleece pullover, September": profit_of("RUN-0002", "P012", "2026-09", workspace.warehouse),
    "Trail water bottle, October": profit_of("RUN-0002", "P001", "2026-10", workspace.warehouse),
    "Camp stove, October": profit_of("RUN-0002", "P007", "2026-10", workspace.warehouse),
}

for name, value in at_close.items():
    print(f"{name:30} {value}")

The camp stove has no figure: its October cost has been missing since milestone 1.

## Step 3: Three files arrive late

They are in their own folder, `landing_late`.

In [ ]:
for file in sorted(workspace.late.glob("*.csv")):
    print(file.name)

### File 1: a refund that is two months late

The file is named for November. Look at the date of the refund inside it.

In [ ]:
late_refund = pd.read_csv(workspace.late / "refunds_2026-11-09.csv")
late_refund

A fleece pullover refund dated **25 September**. Our rule since milestone 2 is that a refund counts in the month of its own date. So it belongs to September, which is closed.

What should it do to September's contribution profit? A refund takes its amount off net sales, and its handling cost is a cost.

In [ ]:
refund_effect = -(late_refund["refund_amount"].iloc[0] + late_refund["return_handling_cost"].iloc[0])

expected_pullover = round(float(at_close["Fleece pullover, September"] + refund_effect), 2)

print("Effect on contribution profit:", refund_effect)
print("Fleece pullover, September should become:", expected_pullover)

A figure for a past month that changes after it was reported is called a **restatement**.

### File 2: a correction

This file has the **same name** as one that was loaded in milestone 5. Compare the two.

In [ ]:
original = pd.read_csv(workspace.places.landing / "shipments_2026-10-19.csv")
corrected = pd.read_csv(workspace.late / "shipments_2026-10-19.csv")

print("Rows:", len(original), "and", len(corrected))

different = original.compare(corrected)
different

One value differs: a carrier cost that was 4.00 is now 9.00. `self` is the original and `other` is the corrected file. Which shipment, and which product?

In [ ]:
row = different.index[0]
corrected.loc[[row]]

In [ ]:
cost_change = corrected.loc[row, "carrier_cost"] - original.loc[row, "carrier_cost"]

expected_bottle = round(float(at_close["Trail water bottle, October"] - cost_change), 2)

print("Shipping cost goes up by:", cost_change)
print("Trail water bottle, October should become:", expected_bottle)

In milestone 5 the loader **rejected** a file like this: same name, different content. That was the safe thing to do then. Now it must **replace** the earlier version.

### File 3: the cost that was missing

In [ ]:
late_cost = pd.read_csv(workspace.late / "product_costs_2026-10_late.csv")
late_cost

The camp stove's October cost, at last. With it, the stove's profit can be worked out. The other terms were already known at close.

In [ ]:
profit = pd.read_parquet(workspace.warehouse / "runs" / "RUN-0002" / "monthly_product_profit.parquet")
stove = profit[(profit["product_id"] == "P007") & (profit["month"] == "2026-10")].iloc[0]

product_cost = stove["units_sold"] * late_cost["unit_cost"].iloc[0]
gross_profit = stove["net_sales"] - product_cost
expected_stove = gross_profit - stove["fulfillment_cost"] - stove["shipping_cost"]
expected_stove = round(float(expected_stove - stove["payment_fees"] - stove["return_cost"]), 2)

print("Product cost:", product_cost)
print("Camp stove, October should become:", expected_stove)

## Step 4: Which months need recalculating?

There are two months of data. Each late file touches only the month of its own records.

In [ ]:
months_touched = {
    "refunds_2026-11-09.csv": late_refund["refund_date"].str[:7].unique().tolist(),
    "shipments_2026-10-19.csv": corrected["ship_date"].str[:7].unique().tolist(),
    "product_costs_2026-10_late.csv": late_cost["month"].unique().tolist(),
}
months_touched

September and October: here, that is everything. But suppose only the refund had arrived. Then only September would need recalculating, and October could be left alone. With two years of data instead of two months, that difference is the whole job.

Recalculating only what changed is called an **incremental run**. It is only safe if it gives exactly what recalculating everything would give.

## Step 5: Save the answer key

In [ ]:
restated_key = {
    "P012 2026-09": {"before": at_close["Fleece pullover, September"], "after": expected_pullover},
    "P001 2026-10": {"before": at_close["Trail water bottle, October"], "after": expected_bottle},
    "P007 2026-10": {"before": None, "after": expected_stove},
    "months": ["2026-09", "2026-10"],
}

save_result("answer_key_restated", restated_key)
restated_key

## What we now know

- A **late record** counts in the month of its own date, even if that month was already reported.
- A **correction** has the same name as a file already loaded, and different content. It must replace the earlier version.
- A figure that changes after it was reported is a **restatement**. We worked out three by hand.
- Each late file touches only certain months. Recalculating only those is an **incremental run**.

**Next:** notebook 2, where the loader and the pipeline do all of this, and you check them against this answer key.